```
┌─────────────────────────────────────────────────────────────────┐
│                    MODELING PIPELINE                            │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  Raw Audio ──→ Preprocessing ──→ Mel Spectrogram ──→ CNN/Transformer
│                     │                                    │      │
│                     ├── Augmentation                     │      │
│                     ├── Quality Filter                   │      │
│                     └── Chunk Splitting                  │      │
│                                                          │      │
│                              ┌────────────────────────────┘     │
│                              ▼                                  │
│                    ┌──── Baseline (EfficientNet-B0) ────┐       │
│                    ├──── Mid-tier (EfficientNet-B3/V2) ─┤       │
│                    ├──── Advanced (BEATs / AST) ────────┤       │
│                    └──── Ensemble + TTA ────────────────┘       │
│                              │                                  │
│                              ▼                                  │
│                    Post-processing + Submission                 │
└─────────────────────────────────────────────────────────────────┘
```

In [11]:
# =============================================================================
# Cell 1: Imports and Configuration
# =============================================================================
"""
This cell serves as the SINGLE SOURCE OF TRUTH for the entire notebook.
- All imports are grouped and documented
- All hyperparameters live in the CFG class (easy to experiment with)
- Reproducibility is enforced via seed_everything()
"""
import os
import gc
import math
import random
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple
# ------------------------------ DATA & VISUALIZATION ------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# ------------------------------ AUDIO PROCESSING ------------------------------
import librosa
import soundfile as sf
# ------------------------------ DEEP LEARNING ------------------------------
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler, autocast  # Mixed precision training
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, OneCycleLR
# ------------------------------ EXTERNAL LIBRARIES ------------------------------
import timm  # Pretrained model zoo (EfficientNetV2, etc.)
import albumentations as A  # Fast image augmentations (applied to spectrograms)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score, f1_score
from tqdm.notebook import tqdm  # Progress bars for notebooks

import colorednoise as cn  # Colored noise augmentation (pip install colorednoise)

warnings.filterwarnings('ignore')  # Cleaner notebook output


# =============================================================================
# Configuration - Single source of truth
# =============================================================================

class CFG:
    """
    Centralized configuration class.
    
    All paths, audio parameters, training hyperparameters, model settings,
    and augmentation parameters are defined here.
    """
    # ========================== PATHS ==========================
    PROJECT_ROOT = Path.cwd().resolve()
    BASE_PATH = PROJECT_ROOT / "kaggle/input/competitions/birdclef-2026"
    TRAIN_CSV = BASE_PATH / "train.csv"
    AUDIO_DIR = BASE_PATH / "train_audio"
    OUTPUT_DIR = Path("/kaggle/working")  # Kaggle working directory

    # ========================== AUDIO SETTINGS ==========================
    SR = 32000  # Sample rate
    DURATION = 5  # Duration of each audio chunk in seconds
    N_SAMPLES = SR * DURATION  # 160000 samples per chunk

    # ========================== MEL SPECTROGRAM ==========================
    # These parameters convert raw audio into a 2D image (mel spectrogram)
    N_MELS = 128  # Number of mel frequency bins (height of image)
    N_FFT = 2048  # FFT window size
    HOP_LENGTH = 512  # Hop length between frames (width of image)
    FMIN = 50  # Minimum frequency (Hz)
    FMAX = 14000  # Maximum frequency (Hz) - most species vocalizations
    POWER = 2.0  # Power for mel spectrogram (2.0 = power, 1.0 = magnitude)
    TOP_DB = 80  # Dynamic range for amplitude-to-dB conversion

    # ========================== TRAINING SETTINGS ==========================
    SEED = 42  # Master random seed for full reproducibility
    N_FOLDS = 5  # Number of folds for Stratified K-Fold CV
    TRAIN_FOLDS = [0, 1, 2, 3]  # Which folds to actually train on (useful for quick tests)
    EPOCHS = 30  # Total training epochs
    BATCH_SIZE = 32  # Batch size (adjust based on GPU memory)
    LR = 1e-3  # Initial learning rate
    MIN_LR = 1e-6  # Minimum learning rate (for schedulers)
    WEIGHT_DECAY = 1e-4  # L2 regularization strength
    WARMUP_EPOCHS = 2  # Number of warmup epochs

    # ========================== MODEL SETTINGS ==========================
    MODEL_NAME = "tf_efficientnetv2_s"  # timm model - excellent speed/accuracy balance
    PRETRAINED = True  # Use ImageNet pretrained weights
    NUM_CLASSES = 206  # Number of species (update after EDA)
    IN_CHANNELS = 1  # 1 = grayscale spectrogram input

    # ========================== AUGMENTATION ==========================
    MIXUP_ALPHA = 0.5  # MixUp alpha parameter
    CUTMIX_ALPHA = 1.0  # CutMix alpha parameter
    MIXUP_PROB = 0.5  # Probability of applying MixUp/CutMix

    # ========================== DATA QUALITY FILTER ==========================
    MIN_RATING = 2.0  # Minimum rating to include training samples

    # ========================== HARDWARE & PERFORMANCE ==========================
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    NUM_WORKERS = 4  # DataLoader workers
    PIN_MEMORY = True  # Faster data transfer to GPU
    USE_AMP = True  # Automatic Mixed Precision (faster + less VRAM)

    # ========================== INFERENCE ==========================
    TTA_STEPS = 5  # Test-Time Augmentation steps


def seed_everything(seed=42):
    """
    Set random seeds across all libraries for full reproducibility.
    
    Args:
        seed (int): Random seed value. Default is 42.
    """
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Make cuDNN deterministic (slower but 100% reproducible)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# =============================================================================
# Initialize everything
# =============================================================================
seed_everything(CFG.SEED)
print(f"🖥️ Device: {CFG.DEVICE}")
print(f"📊 Config loaded:")
print(f"   • Model: {CFG.MODEL_NAME}")
print(f"   • Audio: {CFG.DURATION}s @ {CFG.SR}Hz → {CFG.N_SAMPLES} samples")
print(f"   • Training folds: {CFG.TRAIN_FOLDS}")
print(f"   • Mixed Precision (AMP): {CFG.USE_AMP}")

🖥️ Device: cpu
📊 Config loaded:
   • Model: tf_efficientnetv2_s
   • Audio: 5s @ 32000Hz → 160000 samples
   • Training folds: [0, 1, 2, 3]
   • Mixed Precision (AMP): True


In [12]:
# =============================================================================
# Cell 2: Data Preparation & Fold Splitting
# =============================================================================
"""
This cell performs the critical data preparation steps:
1. Loads the training CSV
2. Creates label encoders (string ↔ integer)
3. Filters low-quality recordings
4. Creates Stratified K-Fold splits
5. Computes class weights to handle severe class imbalance

This cell must be run after Cell 1 (CFG is used extensively here).
"""

# =============================================================================
# 1. Load Training Data
# =============================================================================
df = pd.read_csv(CFG.TRAIN_CSV)
print(f"📥 Loaded training data: {len(df):,} rows × {len(df.columns)} columns")

# =============================================================================
# 2. Build Label Encoder
# =============================================================================
# We convert string labels (e.g. "amegfi") into integer indices (0, 1, 2, ...)
# This is required for PyTorch's CrossEntropyLoss
labels_sorted = sorted(df['primary_label'].unique())
label2idx = {label: idx for idx, label in enumerate(labels_sorted)}
idx2label = {idx: label for label, idx in label2idx.items()}

# Update global config with actual number of classes
CFG.NUM_CLASSES = len(labels_sorted)
print(f"🏷️  Found {CFG.NUM_CLASSES} unique bird species")

# =============================================================================
# 3. Add Useful Columns
# =============================================================================
# label_idx   → integer target for training
# filepath    → full path to the audio file (makes Dataset class cleaner)
df['label_idx'] = df['primary_label'].map(label2idx)
df['filepath'] = df['filename'].apply(lambda x: str(CFG.AUDIO_DIR / x))

# =============================================================================
# 4. Quality Filter (Optional but Recommended)
# =============================================================================
print(f"Before quality filter: {len(df)}")

if 'rating' in df.columns:
    # Keep only recordings with rating >= MIN_RATING (usually 2.0 or 3.0)
    df_filtered = df[df['rating'] >= CFG.MIN_RATING].reset_index(drop=True)
    print(f"After quality filter (rating >= {CFG.MIN_RATING}): {len(df_filtered):,} "
          f"samples" f"({len(df_filtered) / len(df) * 100:.1f}% kept)")
else:
    print("⚠️  No 'rating' column found — skipping quality filter")
    df_filtered = df.copy()

# =============================================================================
# 5. Stratified K-Fold Cross-Validation
# =============================================================================
# We use StratifiedKFold to ensure each fold has roughly the same class distribution.
# This is very important for imbalanced datasets like BirdCLEF. 
skf = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
df_filtered['fold'] = -1  # placeholder

# Assign fold number to each row
# Note: We only assign to val_idx because each row appears in validation set exactly once
for fold, (train_idx, val_idx) in enumerate(skf.split(df_filtered, df_filtered['label_idx'])):
    df_filtered.loc[val_idx, 'fold'] = fold

print(f"\n📊 Classes: {CFG.NUM_CLASSES}")
print(f"📁 Fold distribution:\n{df_filtered['fold'].value_counts().sort_index()}")

# =============================================================================
# 6. Compute Class Weights (Handle Severe Imbalance)
# =============================================================================
# BirdCLEF has extremely imbalanced classes (some species have < 10 recordings,
# others have > 500). We use inverse frequency weighting.
class_counts = df_filtered['label_idx'].value_counts().sort_index().values
class_weights = 1.0 / (class_counts + 1)  # inverse frequency
class_weights = class_weights / class_weights.sum() * CFG.NUM_CLASSES  # normalize
class_weights_tensor = torch.FloatTensor(class_weights).to(CFG.DEVICE)

print(f"\n⚖️  Class weight statistics:")
print(f"   • Min weight : {class_weights.min():.4f}")
print(f"   • Max weight : {class_weights.max():.4f}")
print(f"   • Mean weight: {class_weights.mean():.4f}")
print(f"   • Tensor shape: {class_weights_tensor.shape} → ready for loss function")

📥 Loaded training data: 35,549 rows × 15 columns
🏷️  Found 206 unique bird species
Before quality filter: 35549
After quality filter (rating >= 2.0): 22,411 samples(63.0% kept)

📊 Classes: 206
📁 Fold distribution:
fold
0    4483
1    4482
2    4482
3    4482
4    4482
Name: count, dtype: int64

⚖️  Class weight statistics:
   • Min weight : 0.0498
   • Max weight : 10.3088
   • Mean weight: 1.1135
   • Tensor shape: torch.Size([185]) → ready for loss function


In [15]:
# =============================================================================
# Cell 3: Audio Augmentations (Time-Domain)
# =============================================================================
"""
This cell defines two augmentation classes used during training:

1. AudioAugmentations  → Time-domain augmentations (applied to raw waveform)
2. SpecAugmentations   → Frequency-domain augmentations (applied to mel spectrogram)

These augmentations help the model generalize better to:
- Background noise (Gaussian, pink)
- Recording variations (gain, fade, time shift)
- Bird vocalization variations (pitch shift, time stretch)
- Spectrogram occlusions (SpecAugment-style masking)

All methods are static for easy use inside the Dataset class.
"""


# =============================================================================
# 1. Time-Domain Audio Augmentations
# =============================================================================
class AudioAugmentations:
    """
    Efficient time-domain audio augmentations.
    
    These are applied to the raw waveform BEFORE converting to mel spectrogram.
    They simulate real-world recording variations and help the model become
    robust to noise, volume changes, and slight timing differences.
    """

    @staticmethod
    def add_gaussian_noise(audio: np.ndarray, min_snr_db: float = 5, max_snr_db: float = 20) -> np.ndarray:
        """
        Add white Gaussian noise with random Signal-to-Noise Ratio (SNR).
        
        Args:
            audio: Input waveform (1D numpy array)
            min_snr_db: Minimum SNR in dB (higher = less noise)
            max_snr_db: Maximum SNR in dB
            
        Returns:
            Noisy waveform (float32)
        """
        snr_db = np.random.uniform(min_snr_db, max_snr_db)
        signal_power = np.mean(audio ** 2)
        noise_power = signal_power / (10 ** (snr_db / 10))
        noise = np.random.normal(0, np.sqrt(noise_power), len(audio))
        return audio + noise.astype(np.float32)

    @staticmethod
    def add_pink_noise(audio, min_snr_db=10, max_snr_db=30):
        """
        Add pink noise (1/f noise) - more realistic for natural environments.
        
        Pink noise has equal power per octave (more low-frequency energy).
        This is very common in outdoor bird recordings (wind, distant traffic, etc.).
        """
        snr_db = np.random.uniform(min_snr_db, max_snr_db)
        signal_power = np.mean(audio ** 2)
        noise_power = signal_power / (10 ** (snr_db / 10))
        noise = cn.powerlaw_psd_gaussian(1, len(audio))  # beta=1 for pink
        noise = noise * np.sqrt(noise_power) / (np.std(noise) + 1e-8)
        return audio + noise.astype(np.float32)

    @staticmethod
    def time_shift(audio: np.ndarray, max_shift_pct: float = 0.3) -> np.ndarray:
        """
        Randomly shift the audio in time (circular shift).
        
        Simulates birds singing at slightly different positions in the 5-second window.
        """
        shift = int(len(audio) * np.random.uniform(-max_shift_pct, max_shift_pct))
        return np.roll(audio, shift)

    @staticmethod
    def time_stretch(audio: np.ndarray, rate_range: tuple = (0.8, 1.2)) -> np.ndarray:
        """
        Randomly stretch or compress audio in time (change speed without changing pitch).
        
        Helps the model become invariant to singing speed variations between individuals.
        """
        rate = np.random.uniform(*rate_range)
        stretched = librosa.effects.time_stretch(audio, rate=rate)
        # Resize to original length
        if len(stretched) > len(audio):
            stretched = stretched[:len(audio)]
        else:
            stretched = np.pad(stretched, (0, len(audio) - len(stretched)))
        return stretched

    @staticmethod
    def pitch_shift(audio: np.ndarray, sr: int, n_steps_range: tuple = (-2, 2)) -> np.ndarray:
        """
        Randomly shift pitch up or down (in semitones).
        
        Simulates different bird individuals, ages, or recording equipment variations.
        """
        n_steps = np.random.uniform(*n_steps_range)
        return librosa.effects.pitch_shift(audio, sr=sr, n_steps=n_steps)

    @staticmethod
    def random_gain(audio: np.ndarray, min_gain_db: float = -6, max_gain_db: float = 6) -> np.ndarray:
        """
        Random volume change (in dB).
        
        Makes the model robust to different recording volumes and distances from the bird.
        """
        gain_db = np.random.uniform(min_gain_db, max_gain_db)
        gain = 10 ** (gain_db / 20)
        return audio * gain

    @staticmethod
    def fade(audio: np.ndarray, fade_in_pct: float=0.01, fade_out_pct: float=0.01) -> np.ndarray:
        """
        Apply linear fade-in and fade-out.
        
        Reduces abrupt starts/ends that can occur when cutting 5-second chunks.
        """
        fade_in = int(len(audio) * fade_in_pct)
        fade_out = int(len(audio) * fade_out_pct)
        audio = audio.copy()
        audio[:fade_in] *= np.linspace(0, 1, fade_in)
        audio[-fade_out:] *= np.linspace(1, 0, fade_out)
        return audio


# =============================================================================
# Spectrogram Augmentations (Frequency-Domain)
# =============================================================================

class SpecAugmentations:
    """
    Spectrogram-level augmentations inspired by SpecAugment (Google, 2019).
    
    These are applied AFTER converting audio to mel spectrogram.
    They force the model to learn from partial information (occluded frequencies or time steps).
    """
    @staticmethod
    def _random_mask(spec: torch.Tensor, dim: int, max_mask_pct: float, num_masks: int) -> torch.Tensor:
        """
        Internal helper: randomly masks contiguous regions along a given dimension.
        
        Args:
            spec: Input spectrogram tensor
            dim: Dimension to mask (-2 = frequency, -1 = time)
            max_mask_pct: Maximum percentage of the dimension to mask (0.0 - 1.0)
            num_masks: Number of separate masks to apply
            
        Returns:
            Masked spectrogram (cloned, not in-place)
        """
        spec = spec.clone()
        dim_size = spec.shape[dim]
        
        for _ in range(num_masks):
            mask_size = int(dim_size * np.random.uniform(0, max_mask_pct))
            start = np.random.randint(0, max(1, dim_size - mask_size))
            
            if dim == -2:  # Frequency mask (vertical)
                spec[..., start:start + mask_size, :] = 0
            else:          # Time mask (horizontal)
                spec[..., :, start:start + mask_size] = 0
                
        return spec

    @staticmethod
    def freq_mask(spec: torch.Tensor, max_mask_pct: float=0.15, num_masks: int=2):
        """
        Randomly mask contiguous frequency bands (vertical masks).
        
        Simulates missing frequency information (e.g., due to microphone limitations
        or environmental filtering).
        """
        return SpecAugmentations._random_mask(spec, dim=-2, 
                                              max_mask_pct=max_mask_pct, 
                                              num_masks=num_masks)

    @staticmethod
    def time_mask(spec: torch.Tensor, max_mask_pct: float=0.15, num_masks: int=2):
        """
        Randomly mask contiguous time steps (horizontal masks).
        
        Forces the model to recognize birds even when parts of the call are occluded
        (e.g., overlapping sounds, wind gusts).
        """
        return SpecAugmentations._random_mask(spec, dim=-1, 
                                              max_mask_pct=max_mask_pct, 
                                              num_masks=num_masks)

    @staticmethod
    def mixup(spec1, label1, spec2, label2: torch.Tensor, alpha: float=0.5):
        """
        MixUp augmentation on spectrograms + soft labels.
        
        Creates convex combinations of two examples. Improves generalization
        and calibration. Returns mixed spectrogram and mixed (soft) label.
        """
        lam = np.random.beta(alpha, alpha)
        mixed_spec = lam * spec1 + (1 - lam) * spec2
        mixed_label = lam * label1 + (1 - lam) * label2
        return mixed_spec, mixed_label


print("✅ Augmentation modules ready.")

✅ Augmentation modules ready.


In [ ]:
# =============================================================================
# Cell 4: Dataset Class
# =============================================================================

class BirdCLEFDataset(Dataset):
    """
    Optimized dataset that:
    - Loads audio on-the-fly (memory efficient)
    - Applies time-domain augmentations
    - Converts to mel spectrogram
    - Applies spec augmentations
    - Handles variable-length audio with smart chunking
    """

    def __init__(self, df, cfg, mode='train', augment_audio=True, augment_spec=True):
        self.df = df.reset_index(drop=True)
        self.cfg = cfg
        self.mode = mode
        self.augment_audio = augment_audio and (mode == 'train')
        self.augment_spec = augment_spec and (mode == 'train')
        self.audio_aug = AudioAugmentations()
        self.spec_aug = SpecAugmentations()

        # Precompute mel filterbank for efficiency
        self.mel_basis = librosa.filters.mel(
            sr=cfg.SR, n_fft=cfg.N_FFT, n_mels=cfg.N_MELS,
            fmin=cfg.FMIN, fmax=cfg.FMAX
        )

    def __len__(self):
        return len(self.df)

    def _load_audio(self, filepath):
        """Load and preprocess audio file."""
        try:
            audio, sr = librosa.load(filepath, sr=self.cfg.SR, mono=True)
        except Exception:
            audio = np.zeros(self.cfg.N_SAMPLES, dtype=np.float32)
            return audio

        # Handle duration
        target_len = self.cfg.N_SAMPLES

        if len(audio) < target_len:
            # Pad short audio (tile instead of zero-pad for better training)
            repeats = math.ceil(target_len / len(audio))
            audio = np.tile(audio, repeats)[:target_len]
        elif len(audio) > target_len:
            if self.mode == 'train':
                # Random crop during training
                max_start = len(audio) - target_len
                start = np.random.randint(0, max_start)
                audio = audio[start:start + target_len]
            else:
                # Center crop during validation
                start = (len(audio) - target_len) // 2
                audio = audio[start:start + target_len]

        return audio.astype(np.float32)

    def _audio_to_melspec(self, audio):
        """Convert audio to log-mel spectrogram."""
        S = librosa.feature.melspectrogram(
            y=audio,
            sr=self.cfg.SR,
            n_mels=self.cfg.N_MELS,
            n_fft=self.cfg.N_FFT,
            hop_length=self.cfg.HOP_LENGTH,
            fmin=self.cfg.FMIN,
            fmax=self.cfg.FMAX,
            power=self.cfg.POWER,
        )
        S_db = librosa.power_to_db(S, ref=np.max, top_db=self.cfg.TOP_DB)

        # Normalize to [0, 1]
        S_db = (S_db - S_db.min()) / (S_db.max() - S_db.min() + 1e-8)

        return S_db.astype(np.float32)

    def _apply_audio_augmentations(self, audio):
        """Apply random audio augmentations."""
        if np.random.random() < 0.5:
            audio = self.audio_aug.add_gaussian_noise(audio)
        if np.random.random() < 0.3:
            audio = self.audio_aug.add_pink_noise(audio)
        if np.random.random() < 0.3:
            audio = self.audio_aug.time_shift(audio)
        if np.random.random() < 0.2:
            audio = self.audio_aug.random_gain(audio)
        if np.random.random() < 0.1:
            audio = self.audio_aug.pitch_shift(audio, self.cfg.SR)
        return audio

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        filepath = row['filepath']
        label_idx = row['label_idx']

        # Load audio
        audio = self._load_audio(filepath)

        # Audio augmentations
        if self.augment_audio:
            audio = self._apply_audio_augmentations(audio)

        # Convert to mel spectrogram
        melspec = self._audio_to_melspec(audio)

        # To tensor: (1, n_mels, time_steps)
        melspec = torch.from_numpy(melspec).unsqueeze(0)

        # Spec augmentations
        if self.augment_spec:
            if np.random.random() < 0.5:
                melspec = self.spec_aug.freq_mask(melspec)
            if np.random.random() < 0.5:
                melspec = self.spec_aug.time_mask(melspec)

        # One-hot label for mixup compatibility
        label = torch.zeros(self.cfg.NUM_CLASSES, dtype=torch.float32)
        label[label_idx] = 1.0

        return melspec, label


# Test dataset
train_df = df_filtered[df_filtered['fold'] != 0].reset_index(drop=True)
val_df = df_filtered[df_filtered['fold'] == 0].reset_index(drop=True)

test_ds = BirdCLEFDataset(train_df.head(5), CFG, mode='train')
sample_spec, sample_label = test_ds[0]
print(f"✅ Dataset test passed.")
print(f"   Spectrogram shape: {sample_spec.shape}")
print(f"   Label shape: {sample_label.shape}")
print(f"   Label sum: {sample_label.sum()}")